# Breast Cancer Detection with Logistic Regression + SHAP/LIME Explainability

**Goal:** Classify breast tumors as **Benign** or **Malignant** using the Scikit-learn Breast Cancer Wisconsin (Diagnostic) dataset, and explain *why* the model makes each prediction using SHAP and LIME.

**Tech stack:** Python, Scikit-learn, SHAP, LIME, Matplotlib, Seaborn

**Pipeline:**
1. Load & explore the data
2. Preprocess (train/test split + feature scaling)
3. Train a Logistic Regression classifier
4. Evaluate (Accuracy, Precision, Recall, F1, Confusion Matrix, ROC-AUC)
5. Explain predictions with SHAP (global + local) and LIME (local)

> **Note on environment:** SHAP and LIME are not part of core Scikit-learn. If running locally, first install them:
> ```bash
> pip install shap lime
> ```
> They are pre-installed on Kaggle notebooks.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score
)

import shap
from lime.lime_tabular import LimeTabularExplainer

sns.set_style("whitegrid")
RANDOM_STATE = 11  # fixed seed for reproducibility


## 1. Load the Dataset

The Breast Cancer Wisconsin (Diagnostic) dataset contains 569 samples and 30 numeric features computed from digitized images of a fine needle aspirate (FNA) of a breast mass (e.g. radius, texture, perimeter, area, smoothness, concavity, symmetry, etc.).

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")   # 0 = malignant, 1 = benign
target_names = list(data.target_names)      # ['malignant', 'benign']

print("Shape:", X.shape)
print("Classes:", target_names)
X.head()


## 2. Exploratory Data Analysis

In [ ]:
counts = y.value_counts().sort_index()
labels = [target_names[i] for i in counts.index]

plt.figure(figsize=(5, 4))
plt.bar(labels, counts.values, color=["#e74c3c", "#2ecc71"])
for i, v in enumerate(counts.values):
    plt.text(i, v + 3, str(v), ha="center", fontweight="bold")
plt.title("Class Distribution: Benign vs Malignant")
plt.ylabel("Number of samples")
plt.tight_layout()
plt.show()


In [ ]:
top_features = X.var().sort_values(ascending=False).head(15).index
corr = X[top_features].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True, cbar_kws={"shrink": 0.8})
plt.title("Feature Correlation Heatmap (Top 15 Highest-Variance Features)")
plt.tight_layout()
plt.show()


## 3. Preprocessing: Train/Test Split + Feature Scaling

Logistic Regression is sensitive to feature scale, so we standardize features (zero mean, unit variance). The scaler is fit **only** on the training set to avoid data leakage.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

print("Train shape:", X_train_scaled.shape, " Test shape:", X_test_scaled.shape)


## 4. Train the Logistic Regression Model

In [ ]:
model = LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)
model.fit(X_train_scaled, y_train)
print("Model trained.")


## 5. Evaluate the Model

In [ ]:
y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print(f"Accuracy : {acc:.4f}  ({acc*100:.2f}%)")
print(f"Precision: {prec:.4f}  ({prec*100:.2f}%)")
print(f"Recall   : {rec:.4f}  ({rec*100:.2f}%)")
print(f"F1-Score : {f1:.4f}  ({f1*100:.2f}%)")
print(f"ROC-AUC  : {auc:.4f}  ({auc*100:.2f}%)")
print()
print(classification_report(y_test, y_pred, target_names=target_names))


In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=target_names, yticklabels=target_names,
            cbar=False, annot_kws={"size": 14})
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)

plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, color="#2980b9", linewidth=2, label=f"ROC curve (AUC = {auc:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


## 6. Model Interpretability with SHAP

SHAP (SHapley Additive exPlanations) assigns each feature a contribution value for a given prediction, based on cooperative game theory. Because Logistic Regression is a linear model, we use SHAP's fast, exact `LinearExplainer`.

In [ ]:
explainer = shap.LinearExplainer(model, X_train_scaled)
shap_values = explainer.shap_values(X_test_scaled)

# Global explanation: which features matter most overall, and in which direction
shap.summary_plot(shap_values, X_test_scaled, feature_names=X.columns)


In [ ]:
# Global feature importance (mean absolute SHAP value)
shap.summary_plot(shap_values, X_test_scaled, feature_names=X.columns, plot_type="bar")


In [ ]:
# Local explanation: why did the model predict this for ONE specific patient?
sample_idx = 0
explanation = shap.Explanation(
    values=shap_values[sample_idx],
    base_values=explainer.expected_value,
    data=X_test_scaled.iloc[sample_idx].values,
    feature_names=X.columns,
)
shap.plots.waterfall(explanation)


## 7. Model Interpretability with LIME

LIME (Local Interpretable Model-agnostic Explanations) explains a single prediction by fitting a simple, interpretable model to perturbed samples around that instance.

In [ ]:
lime_explainer = LimeTabularExplainer(
    training_data=X_train_scaled.values,
    feature_names=list(X.columns),
    class_names=target_names,
    mode="classification",
    discretize_continuous=True,
)

sample_idx = 0
instance = X_test_scaled.iloc[sample_idx].values
true_label = target_names[y_test.iloc[sample_idx]]
pred_label = target_names[y_pred[sample_idx]]

exp = lime_explainer.explain_instance(instance, model.predict_proba, num_features=10)

print(f"True label: {true_label}   |   Predicted label: {pred_label}")
exp.as_pyplot_figure()
plt.tight_layout()
plt.show()


In [ ]:
# Inline HTML explanation (readable, great for a portfolio demo / screenshot)
exp.show_in_notebook(show_table=True)


## 8. Conclusion

- The Logistic Regression model achieves **95.61% accuracy** (F1-score ≈ 0.97, ROC-AUC ≈ 0.99) on held-out test data — a strong baseline for a linear, highly interpretable model.
- SHAP's global summary plot shows that features like `worst concave points`, `worst radius`, `worst perimeter`, and `mean concave points` have the largest impact on predictions — consistent with clinical knowledge that irregular, concave cell nuclei boundaries are associated with malignancy.
- LIME confirms these findings at the individual-patient level, giving a transparent, human-readable explanation for each prediction — an important property for any model used to support medical decisions.

**Possible extensions:** try other classifiers (Random Forest, SVM, XGBoost) and compare; hyperparameter tuning with cross-validation; deploy as a simple Streamlit/Flask app with live SHAP explanations.